# Predicting the Unpredictable: Using Monte Carlo to Model Double Pendulum Chaos from Video Data

**Authorship information:** This project is the final submission for Numerical Methods. It builds upon a physical "man-made chaos pendulum" constructed in the Fall '25 semester. The computational framework and Monte Carlo implementation were developed iteratively, using foundational Python libraries such as *NumPy*, *SciPy*, and *Matplotlib*.

### The Challenge: Deterministic Chaos. 
In classical physics, we are taught that if we know the starting position of an object, we can predict its future. However, the Double Pendulum is a famously chaotic system.  While it follows strict laws of gravity and motion, it is so sensitive to its starting conditions that a measurement error smaller than a human hair can result in a completely different trajectory after just a few seconds.

In this project, I bridge the gap between a physical experiment and numerical prediction. By extracting data from video footage of my pendulum, I use Monte Carlo methods to simulate a "cloud" of possible futures, allowing us to see how quickly certainty turns into chaos.

### Project Objectives

In this notebook, I will:

1. **Extract Physical States**: Use computer vision techniques to track red and blue diodes from video footage, converting pixels into angular positions ($\theta$) and velocities ($\omega$).

2. **Define the Physics:** Implement the Lagrangian equations of motion for a compound double pendulum using *scipy.integrate.odeint*.

3. **Propagate Uncertainty:** Apply a Monte Carlo approach by running hundreds of simultaneous simulations, each starting with a slightly different "randomized" version of the initial video data.

4. **Visualize Chaos:** Plot the resulting trajectories to identify the "divergence point"—the moment where the physical system becomes unpredictable.

5. **Analyze Parameters:** (Optional/Hastings) Discuss how varying friction and mass affects the accuracy of the digital model.

### Why Numerical Methods?

The equations governing a double pendulum are non-linear and coupled. This means there is no simple "plug-and-play" formula to find the position at time $t$. We must use numerical integration—breaking time into tiny steps—to solve the system. By adding the Monte Carlo layer, we transition from a single, likely-incorrect guess to a probabilistic forecast.

In [ ]:
!pip install numpy matplotlib scipy

In [ ]:
# import all the necessary packages

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint as int

<p align="center">
  By using the monte carlo equations I will be looking at more than one position. I will be looking <br> at a state (4 variables)
that will impact the position of my chaos pendulum at all times.
</p>

$$\mathbf{y} = \begin{bmatrix} \theta_1 \\ \omega_1 \\ \theta_2 \\ \omega_2 \end{bmatrix} = \begin{bmatrix} \text{Angle of top arm} \\ \text{Speed of top arm} \\ \text{Angle of bottom arm} \\ \text{Speed of bottom arm} \end{bmatrix}$$

# Mathetical Explainations

### The Foundational Eqautions: Lagrangian Formulas

Joseph-Louis Lagrange a italian-french mathetician and physicist who has made extremly impactful contributions to classical mechanics.

The motion equations are derived from the **Lagrangian Mechanics($L$)** (a form of classical mechanics), where the equation is the difference between Kinetic Energy ($T$) and Potential Energy ($V$):
$$L = T - V$$

To find the paths the pendulum takes, we solve the **Euler-Lagrange Equation** for each angle ($\theta_i$):
$$\frac{d}{dt} \left( \frac{\partial L}{\partial \dot{\theta}_i} \right) - \frac{\partial L}{\partial \theta_i} = 0$$

**Euler-Lagrange Equation** is used to determine the path of least action. By using this second-order differential equation I can plug in different angles between the arms of the pendulum to find possible paths that it would be able to take.

These methods are preferred over standard Newtonian force equations because it handles the complex constraints of the two arms in a more organixed fashion as seen below when derived for each individual arm using lagrangian methods.

**For the first arm ($\theta_1$):**
$$(m_1+m_2)L_1\ddot{\theta}_1 + m_2L_2\ddot{\theta}_2\cos(\theta_1-\theta_2) + m_2L_2\dot{\theta}_2^2\sin(\theta_1-\theta_2) + (m_1+m_2)g\sin\theta_1 = 0$$

**For the second arm ($\theta_2$):**
$$m_2L_2\ddot{\theta}_2 + m_2L_1\ddot{\theta}_1\cos(\theta_1-\theta_2) - m_2L_1\dot{\theta}_1^2\sin(\theta_1-\theta_2) + m_2g\sin\theta_2 = 0$$

In [ ]:
# define the function that describes the pendulum's motion. This function will be used in the numerical integration process to compute the pendulum's trajectory over time. The function takes in the current state of the system (angle and angular velocity), time, natural frequency, and damping coefficient, and returns the derivatives of these quantities.

def pendulum(y, t, omega, b):
    theta, omega_dot = y
    dydt = [omega_dot, -b*omega_dot - omega**2 * np.sin(theta)]
    return dydt

In [ ]:
# monte carlo simulation to explore the behavior of the pendulum under different initial conditions and parameters. This involves generating random initial angles and angular velocities, as well as varying the natural frequency and damping coefficient, to see how these factors influence the pendulum's motion. The results can be visualized using plots to show the trajectories of the pendulum over time for different scenarios.

# Set the parameters for the simulation
num_simulations = 100
time = np.linspace(0, 10, 1000)  # Time array from initial_time to final_time with num_points
results = []
for _ in range(num_simulations):
    # Randomly generate initial conditions and parameters
    initial_angle = np.random.uniform(-np.pi, np.pi)  # Initial angle between -pi and pi
    initial_angular_velocity = np.random.uniform(-5, 5)  # Initial angular velocity between -5 and 5
    natural_frequency = np.random.uniform(0.5, 2.0)  # Natural frequency between 0.5 and 2.0
    damping_coefficient = np.random.uniform(0.1, 0.5)  # Damping coefficient between 0.1 and 0.5
    
    # Initial state vector
    y0 = [initial_angle, initial_angular_velocity]
    
    # Integrate the pendulum equations over time
    sol = int(pendulum, y0, time, args=(natural_frequency, damping_coefficient))
    
    # Store the results for plotting
    results.append(sol)

In [ ]:
# Plot the results
plt.figure(figsize=(12, 6))
for sol in results:
    plt.plot(time, sol[:, 0])  # Plot angle over time
plt.title('Pendulum Trajectories for Different Initial Conditions and Parameters')
plt.xlabel('Time (s)')
plt.ylabel('Angle (rad)')
plt.grid()
plt.show()